# Do the reward weights change driving behaviour?

A frozen driving policy can be told how much it should care about crashing. This
notebook checks whether that instruction changes how it drives.

Reward conditioning in GigaFlow and BehaviorBench relies on one policy producing
several driving styles from a set of numbers, so it is worth testing.

Bosch publishes the conditioned policy (`weights/conditioned_ppo.pt`), so there
is nothing to train. A free Colab CPU is enough.


## 1. Set up (about 10 minutes, once per VM)

Clone the simulator, install the evaluation dependencies, and compile the C
environment. `NO_TRAIN=1` skips the training extension, which we don't use.


In [ ]:
%%bash
set -e
cd /content

# simulator and published policy
if [ ! -d behavior-bench ]; then
  git clone --depth 1 -b main-behavior-bench https://github.com/boschresearch/behavior-bench.git
fi

# helper scripts from this repo
if [ ! -d pufferdrive_colab ]; then
  git clone --depth 1 https://github.com/alexanderdbondar/pufferdrive_colab.git
fi

cd behavior-bench

# The SMART planner needs torch-cluster, which is not needed here, so make that
# import optional.
python - <<'PATCH'
p = "pufferlib/planning/__init__.py"
s = open(p).read()
if "SMARTPlanner = SMARTConfig = None" not in s:
    s = s.replace(
        "from .smart import SMARTPlanner, SMARTConfig",
        "try:\n    from .smart import SMARTPlanner, SMARTConfig\nexcept Exception:\n    SMARTPlanner = SMARTConfig = None")
    open(p, "w").write(s)
    print("patched optional SMART import")
PATCH

python -c "import torch" 2>/dev/null || pip install -q torch --index-url https://download.pytorch.org/whl/cpu
pip install -q "numpy<2" "gym==0.23" "gymnasium==0.29.1" "pettingzoo==1.24.1" \
  "shimmy[gym-v21]" scipy pillow msgpack matplotlib tqdm rich rich_argparse psutil imageio torch-geometric

NO_TRAIN=1 python setup.py build_ext --inplace
echo "setup done"


## 2. Get scenes (about 5 minutes, once per VM)

1,000 validation scenes from the Waymo Open Motion Dataset, already converted to
JSON. We turn them into the compact `.bin` maps the simulator reads, then rank
them by how many other agents come near the ego. Safety knobs only matter in
traffic.


In [ ]:
%%bash
set -e
cd /content
if [ ! -f val.zip ]; then
  wget -q -O val.zip https://huggingface.co/datasets/daphne-cornelisse/pufferdrive_womd_val_1000/resolve/main/validation_subset.zip
fi
if [ ! -d data/processed/validation_subset ]; then
  unzip -q -o val.zip
fi
mkdir -p maps/validation
ls data/processed/validation_subset/*.json | wc -l


In [ ]:
%cd /content/behavior-bench
import os, sys
os.environ["DRIVE_BINARIES_DATA_ROOT"] = "/content/maps"
sys.path.insert(0, ".")

from pathlib import Path
from pufferlib.ocean.drive.drive import process_all_maps

if not list(Path("/content/maps/validation").glob("map_*.bin")):
    process_all_maps(Path("/content/data/processed/validation_subset"),
                     Path("/content/maps/validation"), num_workers=2)

!python /content/pufferdrive_colab/conditioning_sweep/make_manifest.py \
  --json-dir /content/data/processed/validation_subset \
  --split-dir /content/maps/validation


## 3. Turn the crash-aversion dial

Run one policy on 40 of the most interactive scenes with five settings of
`alpha_collision` (0 means indifferent to crashing, 3 means strongly
crash-averse). Everything else stays fixed.

Each scene takes about 8 seconds, or roughly 25 minutes on a free Colab CPU.
Results are cached per value, so you can stop and continue.


In [ ]:
%cd /content/behavior-bench
import os
os.environ["DRIVE_BINARIES_DATA_ROOT"] = "/content/maps"

!python /content/pufferdrive_colab/conditioning_sweep/sweep.py \
  --repo /content/behavior-bench \
  --maps 40 \
  --knob alpha_collision \
  --values 0 0.75 1.5 2.25 3 \
  --out /content/runs


In [ ]:
from IPython.display import Image, display
import pandas as pd

display(Image("/content/runs/alpha_collision/plot.png"))
pd.read_csv("/content/runs/alpha_collision/results.csv")


## 4. Try a second dial

`goal_speed` changes the task (how fast the agent wants to go), not just its
caution, so it should move mean speed clearly. A knob that moves nothing is still
a result: the policy never learned to use it.


In [ ]:
%cd /content/behavior-bench
import os
os.environ["DRIVE_BINARIES_DATA_ROOT"] = "/content/maps"

!python /content/pufferdrive_colab/conditioning_sweep/sweep.py \
  --repo /content/behavior-bench \
  --maps 40 \
  --knob goal_speed \
  --values 5 10 20 30 \
  --out /content/runs

from IPython.display import Image, display
import pandas as pd
display(Image("/content/runs/goal_speed/plot.png"))
pd.read_csv("/content/runs/goal_speed/results.csv")


## How to read the result

- A flat line means the dial does nothing. The policy ignores that knob, or it
  never saw it during training.
- A monotone line means the dial works. Behaviour moves in the expected
  direction.
- Safety up with speed unchanged: the policy avoids crashes without simply
  driving slower.
- Safety up with speed down: part of the caution is bought with slower driving.

### What would make this a real finding

1. Sweep more than one knob. A table of which knobs turn, and by how much, is
   more useful than a single curve. GigaFlow did this with a mutual-information
   analysis; a simple monotonicity table is a smaller version.
2. Separate knobs that change the task from knobs that change style.
   `goal_speed` and `delta_goal` change what the agent is trying to do. The
   `alpha_*` weights change how it does it. Mixing them makes style claims weak.
3. Report a negative result. A published conditioned policy that responds to only
   2 of 10 knobs on interactive scenes is a useful finding.
